# Tema 6.2: Casos de uso de segmentación por sector

*Duración estimada: 1 hora*

No todas las máscaras de segmentación nacen iguales. La tolerancia al error cambia drásticamente dependiendo de si estás segmentando una célula cancerígena o contando fresas en un invernadero.

En esta clase, analizaremos cómo el **sector comercial o industrial** determina la configuración de nuestros modelos (como el umbral de confianza en SAM/YOLO) y las métricas que debemos priorizar (Recall vs Precision).

## ⏱️ Estructura de la Clase
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## 1. Conceptos Críticos: Precision vs Recall

Antes de entrar a los sectores, repasemos rápidamente:
- **Falso Positivo (FP)**: El modelo predice que hay algo, pero no lo hay. (Inventa).
- **Falso Negativo (FN)**: El modelo dice que no hay nada, pero sí lo había. (Omite).

- **Precision (Precisión)**: De todo lo que el modelo marcó como positivo, ¿cuánto era real? (Penaliza los Falsos Positivos).
- **Recall (Exhaustividad/Sensibilidad)**: De todos los positivos reales en el mundo, ¿cuántos logró atrapar el modelo? (Penaliza los Falsos Negativos).

## 2. Sector Salud: Segmentación Médica (Tumores/Lesiones)

**Regla:** El Recall es Rey.

- **El problema del Falso Negativo:** Si el modelo omite un tumor (FN), el paciente no recibe tratamiento.
- **El problema del Falso Positivo:** Si el modelo marca tejido sano como tumor (FP), el médico lo revisará y lo descartará.
- **Métricas:** Coeficiente Dice y Recall altísimo.
- **Umbral de Confianza:** Se ajusta a la baja (ej. 0.25) para sobre-segmentar.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# --- SIMULACIÓN: SECTOR SALUD ---
imagen_medica = np.zeros((200, 200), dtype=np.uint8)
# Ground Truth (Tumor real - pequeño)
tumor_real = np.zeros((200, 200), dtype=bool)
cv2.circle(tumor_real.view(np.uint8), (100, 100), 15, 1, -1)

# Predicción (El modelo predice una zona más grande para no fallar - High Recall)
tumor_predicho = np.zeros((200, 200), dtype=bool)
cv2.circle(tumor_predicho.view(np.uint8), (100, 100), 25, 1, -1)

def metricas(y_true, y_pred):
    intersection = np.logical_and(y_true, y_pred).sum()
    recall = intersection / y_true.sum()
    precision = intersection / y_pred.sum()
    dice = (2. * intersection) / (y_true.sum() + y_pred.sum())
    return recall, precision, dice

rec, prec, dice = metricas(tumor_real, tumor_predicho)
print(f"Salud - Recall: {rec*100:.1f}% (¡Excelente, no se nos escapó el tumor!)")
print(f"Salud - Precision: {prec*100:.1f}% (Baja, segmentamos tejido sano de más)")
print(f"Salud - Dice Score: {dice:.3f}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
ax1.imshow(tumor_real, cmap='Blues'); ax1.set_title('Ground Truth (Tumor)')
ax2.imshow(tumor_predicho, cmap='Oranges'); ax2.set_title('Predicción (High Recall)')
ax3.imshow(tumor_predicho ^ tumor_real, cmap='Reds'); ax3.set_title('Exceso (Falsos Positivos)')
plt.show()

## 3. Sector Manufactura: Control de Calidad Automático

**Regla:** Balance estricto o Precision priorizada.

- **Contexto:** Si el modelo detecta un defecto, un brazo robótico tira la pieza a la basura.
- **El problema del Falso Positivo:** Tirar piezas buenas a la basura (pérdida de miles de dólares).
- **El problema del Falso Negativo:** Un producto defectuoso llega al cliente.
- **Umbral de Confianza:** Muy alto (ej. 0.85). Solo descartar si el modelo está 100% seguro.

In [ ]:
# --- SIMULACIÓN: SECTOR MANUFACTURA ---
confianza_modelo = np.array([
    [0.1, 0.2, 0.1],
    [0.3, 0.9, 0.4],
    [0.1, 0.5, 0.1]
])

def control_calidad(matriz_confianza, threshold):
    # Generamos la máscara solo donde la confianza supere el threshold
    mascara_defecto = matriz_confianza >= threshold
    return mascara_defecto

# Si usamos un umbral bajo (0.4), detectamos 3 zonas como defectuosas
mascara_umbral_bajo = control_calidad(confianza_modelo, 0.4)
# Si usamos un umbral alto (0.85), solo detectamos la zona donde estamos seguros
mascara_umbral_alto = control_calidad(confianza_modelo, 0.85)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
ax1.imshow(confianza_modelo, cmap='gray', vmin=0, vmax=1); ax1.set_title('Mapa Calor Confianza')
ax2.imshow(mascara_umbral_bajo, cmap='Reds'); ax2.set_title('Threshold 0.4 (Tira piezas buenas)')
ax3.imshow(mascara_umbral_alto, cmap='Greens'); ax3.set_title('Threshold 0.85 (Alta Precision)')
plt.show()

In [ ]:
# Gráfico de Trade-off Original de la clase
thresholds = np.linspace(0.1, 0.9, 9)
recall_manufactura = [100, 95, 90, 80, 70, 50, 30, 15, 5]
precision_manufactura = [30, 40, 50, 65, 80, 92, 97, 99, 100]

plt.figure(figsize=(10, 4))
plt.plot(thresholds, recall_manufactura, label='Recall (Encontrar todos los defectos)', marker='o', color='red')
plt.plot(thresholds, precision_manufactura, label='Precision (Evitar tirar chips buenos)', marker='s', color='blue')
plt.title('Trade-off Precision/Recall vs Umbral de Confianza (Manufactura)')
plt.xlabel('Umbral de Confianza del Modelo')
plt.ylabel('Puntuación (%)')
plt.axvline(x=0.7, color='gray', linestyle='--', label='Punto Dulce')
plt.legend(); plt.grid(True); plt.show()

## 4. Sector Agricultura: Conteo y Cosecha

**Caso de uso:** Robot cosechador de manzanas.

Aquí nos enfrentamos a desafíos físicos (oclusión). Las manzanas se tapan entre sí por las hojas. 
- **El reto:** ¿Cómo diferenciar 3 manzanas pegadas de 1 manzana gigante?
- **La solución:** Segmentación por Instancia (Masks) sobre Segmentación Semántica.

In [ ]:
# --- SIMULACIÓN: SECTOR AGRICULTURA ---
# Tres manzanas traslapadas
img_agricultura = np.zeros((200, 300), dtype=np.uint8)
mask1 = np.zeros((200, 300), dtype=bool)
mask2 = np.zeros((200, 300), dtype=bool)
mask3 = np.zeros((200, 300), dtype=bool)

cv2.circle(mask1.view(np.uint8), (100, 100), 40, 1, -1)
cv2.circle(mask2.view(np.uint8), (140, 120), 40, 1, -1)
cv2.circle(mask3.view(np.uint8), (180, 90),  40, 1, -1)

# La segmentación Semántica lo ve como un solo bloque masivo (No sirve para contar)
semantica = mask1 | mask2 | mask3

# La segmentación por Instancia (SAM/YOLO) mantiene IDs separados
instancia = np.zeros((200, 300), dtype=np.uint8)
instancia[mask1] = 1 # Manzana ID 1
instancia[mask2] = 2 # Manzana ID 2
instancia[mask3] = 3 # Manzana ID 3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(semantica, cmap='Reds'); ax1.set_title('Seg. Semántica (Conteo = 1 gigante)')
ax2.imshow(instancia, cmap='viridis'); ax2.set_title('Seg. Instancia (Conteo = 3)')
plt.show()

## 5. Ejercicio de Debate en Clase

Imagina que estás diseñando el sistema de percepción visual para un coche autónomo (Tesla/Waymo).

**Pregunta:** Estás detectando **peatones** en la noche. ¿Preferirías que tu modelo sufra de Falsos Positivos (detectar un buzón de correo como si fuera un humano y frenar) o de Falsos Negativos (no ver al peatón)? 

*Responde usando los conceptos de Precision y Recall discutidos arriba y justifica tu elección de umbral.*